# Training experiments

Launcher for the MGMT training pipeline: it picks a backbone and a validation fold, runs each experiment below through `python -m training.run train`, and reads the run folders back. Training logic lives in `training/`, so everything here is settings and reading results. Open it from `notebooks/` or the repository root using the repo environment (`requirements.txt`).

Each experiment is a list of `--set key=value` overrides on `configs/train.yaml` and `configs/backbones/<backbone>.yaml`. A `smoke` run trains on the validation patients to show the pipeline runs end to end; its record is marked invalid for results. A real run needs `macenko_reference_patch_ids` set for the fold. Values for Experiment A (tiles per patient) and Experiment B (JPEG quality) are not decided yet, so they are not listed. Add them as rows when they are.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "training").is_dir() else Path.cwd().parent
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
from training import report

In [ ]:
BACKBONE = "swin_v2_t"            # swin_v2_t or convnext_v2_t
FOLD = 0                           # validation fold, 0 to 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPERIMENTS = {
    "smoke": ["smoke_train_on_validation=true", "tiles_per_bag=8",
              "validation_tiles_per_patient=64", "max_eval_tiles_per_patient=64",
              "stages_real=[{learning_rate: 1.0e-5, epochs: 2, class_weighted: true}]"],
    "baseline": [],
}
SELECTED = ["smoke"]

In [ ]:
runs = {name: report.launch(BACKBONE, FOLD, EXPERIMENTS[name], DEVICE) for name in SELECTED}
runs

In [ ]:
report.summarize(runs)

In [ ]:
figure = report.plot_curves(runs)
figure.savefig(REPO / "data" / "runs" / f"curves_{BACKBONE}_fold{FOLD}.png", dpi=150)

AUROC here is computed on the validation fold's patients from `predictions.csv`. It is one fold of about 30 patients, so it is a sanity check on the pipeline and not a result. Thresholded metrics and bootstrap intervals are not built yet because the threshold rule is not decided.